In [2]:
!nvidia-smi

Sat Jun 13 10:37:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.4 MB/s eta 0:00:00


In [4]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 52.1/118.9 GB disk)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Results will be saved to:", SAVE_DIR)

Results will be saved to: /content/drive/MyDrive/YOLO12n_VisDrone_70epochs


In [ ]:
from ultralytics import YOLO
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")
RUN_NAME = "train_yolo12n_visdrone_70"

SAVE_DIR.mkdir(parents=True, exist_ok=True)

run_dir = SAVE_DIR / RUN_NAME
last_ckpt = run_dir / "weights" / "last.pt"

# If previous training stopped, resume from last.pt
if last_ckpt.exists():
    print("Checkpoint found. Resuming training from:")
    print(last_ckpt)

    model = YOLO(str(last_ckpt))
    train_results = model.train(resume=True)

else:
    print("No checkpoint found. Starting new YOLO12n training...")

    model = YOLO("yolo12n.pt")

    train_results = model.train(
        data="VisDrone.yaml",
        epochs=70,
        imgsz=640,
        batch=-1,
        device=0,
        workers=2,
        plots=True,

        # important
        save=True,
        save_period=5,

        project=str(SAVE_DIR),
        name=RUN_NAME,
        exist_ok=True
    )

Checkpoint found. Resuming training from:
/content/drive/MyDrive/YOLO12n_VisDrone_70epochs/train_yolo12n_visdrone_70/weights/last.pt
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=14, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode

# Checkpoint if Model Crashes or Timed **out**::****

# Step 5: Check dataset downloaded **correctly**

In [ ]:
from pathlib import Path
from ultralytics.utils import SETTINGS

dataset_path = Path(SETTINGS["datasets_dir"]) / "VisDrone"

print("Dataset path:", dataset_path)
print("Exists:", dataset_path.exists())

for split in ["train", "val", "test"]:
    img_dir = dataset_path / "images" / split
    lbl_dir = dataset_path / "labels" / split

    image_count = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0
    label_count = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    print(f"{split}: images={image_count}, labels={label_count}")

# Step 6: Validate on validation **set**

In [ ]:
from ultralytics import YOLO
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")
RUN_NAME = "train_yolo12n_visdrone_70"

best_model_path = SAVE_DIR / RUN_NAME / "weights" / "best.pt"

model = YOLO(str(best_model_path))

val_metrics = model.val(
    data="VisDrone.yaml",
    split="val",
    imgsz=640,
    batch=-1,
    device=0,
    plots=True,
    save_json=True,
    project=str(SAVE_DIR),
    name="val_results",
    exist_ok=True
)

print("Validation Precision:", val_metrics.box.mp)
print("Validation Recall:", val_metrics.box.mr)
print("Validation mAP@50:", val_metrics.box.map50)
print("Validation mAP@50-95:", val_metrics.box.map)

# **Step 7: Test on test set**

In [ ]:
test_metrics = model.val(
    data="VisDrone.yaml",
    split="test",
    imgsz=640,
    batch=-1,
    device=0,
    plots=True,
    save_json=True,
    project=str(SAVE_DIR),
    name="test_results",
    exist_ok=True
)

print("Test Precision:", test_metrics.box.mp)
print("Test Recall:", test_metrics.box.mr)
print("Test mAP@50:", test_metrics.box.map50)
print("Test mAP@50-95:", test_metrics.box.map)

# **Step 8: Save final metrics into CSV**

In [ ]:
import pandas as pd
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")

summary = {
    "model": ["YOLO12n"],
    "dataset": ["VisDrone"],
    "epochs": [70],
    "image_size": [640],

    "val_precision": [val_metrics.box.mp],
    "val_recall": [val_metrics.box.mr],
    "val_mAP50": [val_metrics.box.map50],
    "val_mAP50_95": [val_metrics.box.map],

    "test_precision": [test_metrics.box.mp],
    "test_recall": [test_metrics.box.mr],
    "test_mAP50": [test_metrics.box.map50],
    "test_mAP50_95": [test_metrics.box.map],
}

df = pd.DataFrame(summary)

metrics_csv = SAVE_DIR / "final_metrics_summary.csv"
df.to_csv(metrics_csv, index=False)

print("Saved final metrics to:", metrics_csv)
df

# Step 9: Save per-class mAP **results**

In [ ]:
import pandas as pd
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")

class_names = model.names

val_per_class = pd.DataFrame({
    "class_id": list(class_names.keys()),
    "class_name": list(class_names.values()),
    "val_mAP50_95": val_metrics.box.maps
})

test_per_class = pd.DataFrame({
    "class_id": list(class_names.keys()),
    "class_name": list(class_names.values()),
    "test_mAP50_95": test_metrics.box.maps
})

val_per_class_path = SAVE_DIR / "val_per_class_map.csv"
test_per_class_path = SAVE_DIR / "test_per_class_map.csv"

val_per_class.to_csv(val_per_class_path, index=False)
test_per_class.to_csv(test_per_class_path, index=False)

print("Saved:", val_per_class_path)
print("Saved:", test_per_class_path)

display(val_per_class)
display(test_per_class)

# **Step 10: Show confusion matrix, PR curve, recall curve, F1 curve**

In [ ]:
from IPython.display import Image, display
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")

val_dir = SAVE_DIR / "val_results"
test_dir = SAVE_DIR / "test_results"

plot_files = [
    val_dir / "confusion_matrix.png",
    val_dir / "confusion_matrix_normalized.png",
    val_dir / "PR_curve.png",
    val_dir / "P_curve.png",
    val_dir / "R_curve.png",
    val_dir / "F1_curve.png",

    test_dir / "confusion_matrix.png",
    test_dir / "confusion_matrix_normalized.png",
    test_dir / "PR_curve.png",
    test_dir / "P_curve.png",
    test_dir / "R_curve.png",
    test_dir / "F1_curve.png",
]

for p in plot_files:
    if p.exists():
        print("\nShowing:", p)
        display(Image(filename=str(p)))
    else:
        print("Missing:", p)

# Step 11: Show training result **graph**

In [ ]:
from IPython.display import Image, display
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")
RUN_NAME = "train_yolo12n_visdrone_70"

results_png = SAVE_DIR / RUN_NAME / "results.png"
results_csv = SAVE_DIR / RUN_NAME / "results.csv"

if results_png.exists():
    display(Image(filename=str(results_png)))
else:
    print("results.png not found")

print("Training CSV:", results_csv)

# Step 12: Predict sample test **images**

In [ ]:
from ultralytics.utils import SETTINGS
from pathlib import Path

dataset_path = Path(SETTINGS["datasets_dir"]) / "VisDrone"
test_img_dir = dataset_path / "images" / "test"

sample_images = list(test_img_dir.glob("*.jpg"))[:10]

pred_results = model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(SAVE_DIR),
    name="sample_predictions",
    exist_ok=True
)

print("Sample predictions saved to:", SAVE_DIR / "sample_predictions")

# Show predictions:

In [ ]:
from IPython.display import Image, display
from pathlib import Path

pred_dir = SAVE_DIR / "sample_predictions"

for p in list(pred_dir.glob("*.jpg"))[:10]:
    display(Image(filename=str(p)))

# Step 13: Zip all **results**

In [ ]:
import shutil
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")

zip_base = "/content/yolo12n_visdrone_70epochs_results"
zip_file = shutil.make_archive(zip_base, "zip", SAVE_DIR)

print("ZIP created:", zip_file)

# Copy ZIP to Drive:

In [ ]:
import shutil
from pathlib import Path

src_zip = Path("/content/yolo12n_visdrone_70epochs_results.zip")
dst_zip = Path("/content/drive/MyDrive/yolo12n_visdrone_70epochs_results.zip")

shutil.copy2(src_zip, dst_zip)

print("ZIP copied to:", dst_zip)

In [ ]:
!pip install -U sahi pycocotools tqdm opencv-python-headless

In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics.utils import SETTINGS

SAVE_DIR = Path("/content/drive/MyDrive/YOLO12n_VisDrone_70epochs")
RUN_NAME = "train_yolo12n_visdrone_70"

BEST_MODEL = SAVE_DIR / RUN_NAME / "weights" / "best.pt"

dataset_path = Path(SETTINGS["datasets_dir"]) / "VisDrone"

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Load YOLO once to get class names
yolo_model = YOLO(str(BEST_MODEL))
class_names = yolo_model.names

print("Best model:", BEST_MODEL)
print("Dataset path:", dataset_path)
print("Device:", device)
print("Classes:", class_names)

# SAHI wrapper for Ultralytics YOLO
sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(BEST_MODEL),
    confidence_threshold=0.001,   # low threshold for mAP calculation
    device=device
)

In [ ]:
import json
from PIL import Image
from tqdm import tqdm

def create_coco_gt_from_yolo(split="val"):
    img_dir = dataset_path / "images" / split
    lbl_dir = dataset_path / "labels" / split

    out_dir = SAVE_DIR / f"sahi_{split}_evaluation"
    out_dir.mkdir(parents=True, exist_ok=True)

    gt_json_path = out_dir / f"{split}_gt_coco.json"

    image_paths = sorted(list(img_dir.glob("*.jpg")))

    images = []
    annotations = []
    categories = []

    for cls_id, cls_name in class_names.items():
        categories.append({
            "id": int(cls_id) + 1,
            "name": str(cls_name),
            "supercategory": "object"
        })

    ann_id = 1
    image_id_map = {}

    for image_id, img_path in enumerate(tqdm(image_paths, desc=f"Creating COCO GT for {split}"), start=1):
        img = Image.open(img_path)
        W, H = img.size

        image_id_map[str(img_path)] = image_id

        images.append({
            "id": image_id,
            "file_name": img_path.name,
            "width": W,
            "height": H
        })

        label_path = lbl_dir / f"{img_path.stem}.txt"

        if not label_path.exists():
            continue

        lines = label_path.read_text().strip().splitlines()

        for line in lines:
            if not line.strip():
                continue

            parts = line.split()
            if len(parts) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, parts)

            cls = int(cls)

            x = (xc - bw / 2) * W
            y = (yc - bh / 2) * H
            w = bw * W
            h = bh * H

            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": cls + 1,
                "bbox": [x, y, w, h],
                "area": w * h,
                "iscrowd": 0
            })

            ann_id += 1

    coco_gt = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    with open(gt_json_path, "w") as f:
        json.dump(coco_gt, f)

    print("Saved COCO GT:", gt_json_path)
    print("Images:", len(images))
    print("Annotations:", len(annotations))

    return gt_json_path, image_paths, image_id_map

In [ ]:
def bbox_to_xywh(obj_prediction):
    bbox = obj_prediction.bbox

    try:
        x, y, w, h = bbox.to_xywh()
    except:
        x = bbox.minx
        y = bbox.miny
        w = bbox.maxx - bbox.minx
        h = bbox.maxy - bbox.miny

    return [float(x), float(y), float(w), float(h)]


def run_sahi_on_split(
    split="val",
    slice_height=512,
    slice_width=512,
    overlap=0.20,
    max_images=None,
    save_visuals_count=20
):
    gt_json_path, image_paths, image_id_map = create_coco_gt_from_yolo(split=split)

    if max_images is not None:
        image_paths = image_paths[:max_images]

    out_dir = SAVE_DIR / f"sahi_{split}_evaluation"
    visual_dir = out_dir / f"visuals_{slice_width}x{slice_height}_overlap{overlap}"
    visual_dir.mkdir(parents=True, exist_ok=True)

    pred_json_path = out_dir / f"{split}_sahi_predictions_{slice_width}x{slice_height}_overlap{overlap}.json"

    coco_predictions = []

    for idx, img_path in enumerate(tqdm(image_paths, desc=f"SAHI prediction on {split}")):
        result = get_sliced_prediction(
            image=str(img_path),
            detection_model=sahi_model,
            slice_height=slice_height,
            slice_width=slice_width,
            overlap_height_ratio=overlap,
            overlap_width_ratio=overlap,
            perform_standard_pred=True,
            postprocess_type="NMS",
            postprocess_match_metric="IOU",
            postprocess_match_threshold=0.5,
            verbose=0
        )

        image_id = image_id_map[str(img_path)]

        for obj in result.object_prediction_list:
            bbox_xywh = bbox_to_xywh(obj)

            try:
                score = float(obj.score.value)
            except:
                score = float(obj.score)

            category_id = int(obj.category.id) + 1

            coco_predictions.append({
                "image_id": image_id,
                "category_id": category_id,
                "bbox": bbox_xywh,
                "score": score
            })

        if idx < save_visuals_count:
            result.export_visuals(
                export_dir=str(visual_dir),
                file_name=img_path.stem
            )

    with open(pred_json_path, "w") as f:
        json.dump(coco_predictions, f)

    print("Saved SAHI predictions:", pred_json_path)
    print("Total predictions:", len(coco_predictions))
    print("Visual samples saved to:", visual_dir)

    return gt_json_path, pred_json_path

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import pandas as pd

def evaluate_coco_map(gt_json_path, pred_json_path, split="val", tag="sahi"):
    coco_gt = COCO(str(gt_json_path))

    with open(pred_json_path, "r") as f:
        preds = json.load(f)

    if len(preds) == 0:
        print("No predictions found. Cannot evaluate.")
        return None

    coco_dt = coco_gt.loadRes(str(pred_json_path))

    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.params.maxDets = [1, 10, 300]

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    stats = coco_eval.stats

    metrics = {
        "split": split,
        "method": tag,
        "mAP50_95": stats[0],
        "mAP50": stats[1],
        "mAP75": stats[2],
        "mAP_small": stats[3],
        "mAP_medium": stats[4],
        "mAP_large": stats[5],
        "AR_1": stats[6],
        "AR_10": stats[7],
        "AR_300": stats[8],
        "AR_small": stats[9],
        "AR_medium": stats[10],
        "AR_large": stats[11],
    }

    df = pd.DataFrame([metrics])

    out_csv = SAVE_DIR / f"{split}_{tag}_coco_metrics.csv"
    df.to_csv(out_csv, index=False)

    print("Saved metrics:", out_csv)

    return df

In [ ]:
gt_json, pred_json = run_sahi_on_split(
    split="val",
    slice_height=512,
    slice_width=512,
    overlap=0.20,
    max_images=None,
    save_visuals_count=20
)

sahi_val_metrics = evaluate_coco_map(
    gt_json,
    pred_json,
    split="val",
    tag="yolo12n_sahi_512"
)

sahi_val_metrics

In [ ]:
BEST_SLICE_HEIGHT = 512
BEST_SLICE_WIDTH = 512
BEST_OVERLAP = 0.20

gt_json, pred_json = run_sahi_on_split(
    split="test",
    slice_height=BEST_SLICE_HEIGHT,
    slice_width=BEST_SLICE_WIDTH,
    overlap=BEST_OVERLAP,
    max_images=None,
    save_visuals_count=20
)

sahi_test_metrics = evaluate_coco_map(
    gt_json,
    pred_json,
    split="test",
    tag=f"yolo12n_sahi_{BEST_SLICE_WIDTH}"
)

sahi_test_metrics

In [ ]:
baseline_vs_sahi = pd.DataFrame([
    {
        "method": "YOLO12n baseline",
        "val_precision": val_metrics.box.mp,
        "val_recall": val_metrics.box.mr,
        "val_mAP50": val_metrics.box.map50,
        "val_mAP50_95": val_metrics.box.map,
        "test_precision": test_metrics.box.mp,
        "test_recall": test_metrics.box.mr,
        "test_mAP50": test_metrics.box.map50,
        "test_mAP50_95": test_metrics.box.map,
    },
    {
        "method": "YOLO12n + SAHI",
        "val_precision": None,
        "val_recall": None,
        "val_mAP50": float(sahi_val_metrics["mAP50"].iloc[0]),
        "val_mAP50_95": float(sahi_val_metrics["mAP50_95"].iloc[0]),
        "test_precision": None,
        "test_recall": None,
        "test_mAP50": float(sahi_test_metrics["mAP50"].iloc[0]),
        "test_mAP50_95": float(sahi_test_metrics["mAP50_95"].iloc[0]),
    }
])

comparison_csv = SAVE_DIR / "baseline_vs_sahi_comparison.csv"
baseline_vs_sahi.to_csv(comparison_csv, index=False)

print("Saved:", comparison_csv)
baseline_vs_sahi